In [1]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

In [2]:
import optuna
import pickle
from functools import partial
from pathlib import Path

from simulator.simulation.modules import Campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check

/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

In [4]:
from simulator.model.rlb_dp_bidder import RLBDPBidder

In [5]:
auction_mode = "FPA"  # or "VCG"
best_params_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"
best_models_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"

# metric to optimize: CPC_REL / RMSE / SCR
metric = "SCR"
n_trials = 10

In [6]:
data_config = {
    "train": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_train_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_train_final.csv",
    },
    "test": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_test_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_test_final.csv",
    },
}

data_config

{'train': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_train_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_train_final.csv'},
 'test': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_test_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_test_final.csv'}}

In [7]:
stats_path = data_config['train']['stats_path']
campaigns_path = data_config['train']['campaigns_path']

In [8]:
stats_df = pd.read_csv(stats_path)

In [9]:
def objective_rlb_dp(trial, metric='RMSE_T', auction_mode='FPA'):

    max_bid = trial.suggest_float('max_bid', 10, 500, log=True)
    gamma = trial.suggest_float('gamma', 0.80, 1.00) 
    N_bound = trial.suggest_int('N_bound', 6, 72)
    B_bound = trial.suggest_int('B_bound', 1e3, 2e4, log=True)

    custom_params = {
        "max_bid": max_bid,
        "gamma": gamma,
        "model_path": None,
        "N_bound": N_bound,
        "B_bound": B_bound,
    }


    bidder = RLBDPBidder(custom_params)

    bidder.fit(stats_df)
    import os, uuid

    os.makedirs("tmp_models", exist_ok=True)
    TMP_MODEL_PATH = f"tmp_models/rlb_dp_trial{trial.number}_{uuid.uuid4().hex}.pkl"
    bidder.save_model(TMP_MODEL_PATH)

    # прогоняем через тот же пайплайн проверки
    res = autobidder_check(
        bidder=RLBDPBidder,
        params={
            "input_campaigns": campaigns_path,
            "input_stats": stats_path,
            "max_bid": max_bid,
            "gamma": gamma,
            "model_path": TMP_MODEL_PATH,
            "N_bound": N_bound,
            "B_bound": B_bound,
        },
        auction_mode=auction_mode,
    )

    print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")
    if metric == 'RMSE_T':
        return res['score'][1]
    elif metric == 'CPC_REL':
        return res['score'][0]
    elif metric == 'SCR':
        return res['score'][2]


def opt_search_rlb_dp(n_trials, metric='RMSE_T', auction_mode='FPA'):
    study = optuna.create_study(
        direction='maximize' if metric == 'SCR' else 'minimize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    
    study.optimize(
        partial(objective_rlb_dp, metric=metric, auction_mode=auction_mode),
        n_trials=n_trials,
        n_jobs=6
    )

    print('Best trial:')
    trial = study.best_trial
    print(f'  Value: {trial.value}')
    print('  Params: ')

    dict_path = f'best_params/rlb_dp_{metric.lower()}_{auction_mode}.pkl'
    params_dict = {}
    for key, value in trial.params.items():
        print(f'    {key}: {value}')
        params_dict[key] = value

    with open(dict_path, 'wb') as f:
        pickle.dump(params_dict, f)

    return study


def train_best_rlb_dp(best_params_path, model_path='rlb_dp_model_tuned.pkl'):
    """Обучить и сохранить модель с лучшими параметрами (после optuna)."""
    with open(best_params_path, 'rb') as f:
        best_params = pickle.load(f)

    custom_params = {
        "max_bid": best_params["max_bid"],
        "gamma": best_params["gamma"],
        "model_path": None,
        "N_bound": best_params["N_bound"],
        "B_bound": best_params["B_bound"],
    }

    bidder = RLBDPBidder(custom_params)
    bidder.fit(stats_df)
    bidder.save_model(model_path)
    return bidder


In [ ]:
study_rlb = opt_search_rlb_dp(n_trials, metric, auction_mode)

[I 2026-04-28 01:26:08,289] A new study created in memory with name: no-name-7ef72277-eb4c-4aa1-a4e5-bdc80085b6a9
Hours:  11%|█         | 4/37 [00:00<00:04,  8.17it/s]









Hours:  14%|█▎        | 5/37 [00:01<00:05,  6.04it/s]









Hours:  16%|█▌        | 6/37 [00:01<00:08,  3.73it/s]







Hours: 100%|██████████| 35/35 [00:00<00:00, 42.01it/s] 




Hours:  22%|██▏       | 8/37 [00:01<00:08,  3.56it/s]




Hours:  24%|██▍       | 9/37 [00:02<00:07,  3.71it/s]




Hours:  27%|██▋       | 10/37 [00:02<00:06,  4.19it/s]




Hours:  30%|██▉       | 11/37 [00:02<00:05,  4.64it/s]




Hours:  32%|███▏      | 12/37 [00:02<00:05,  4.99it/s]




Hours:  35%|███▌      | 13/37 [00:02<00:04,  5.27it/s]




Hours:  38%|███▊      | 14/37 [00:03<00:04,  5.52it/s]




Hours: 100%|██████████| 40/40 [00:02<00:00, 13.48it/s]




Hours:  43%|████▎     | 16/37 [00:03<00:03,  6.09it/s]




Hours:  46%|████▌     | 17/37 [00:03<00:03,  5.34it/s]




Hours:  49%|████▊     | 18/37 [00:03<00:03,  4.93it

In [ ]:
# best_params_path = f'best_params/{best_params_subfolder}/{metric.lower()}.pkl'
best_params_path='best_params/rlb_dp_scr_FPA.pkl'
best_model_path = f'best_models/{best_models_subfolder}/{metric.lower()}.pkl'

rlb_bidder_best = train_best_rlb_dp(best_params_path, best_model_path)

Hours: 100%|██████████| 51/51 [00:01<00:00, 42.06it/s]


In [ ]:
best_params_path

'best_params/rlb_dp_scr_FPA.pkl'

In [ ]:
best_params_rlb = pd.read_pickle(best_params_path)
best_model_rlb_path = best_model_path

In [ ]:
campaigns_path_test = data_config['test']['campaigns_path']
stats_path_test = data_config['test']['stats_path']

In [ ]:
res = autobidder_check(
    bidder=RLBDPBidder,
    params = {
        "input_campaigns": campaigns_path_test,
        "input_stats": stats_path_test,
        "model_path": best_model_path,
        **best_params_rlb
    },
    auction_mode=auction_mode,
)

In [ ]:
print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")

CPC_REL: 328.5357522422687, rmse: 1.3405143675549405, SCR: 32396.07802075457
